Notebook to merge CSV files and create the final project csv file by Andrew Miller

#1 Setup Environment, Load Libraries and Create File Paths

In [1]:
# Import necessary libraries
import pandas as pd
from pathlib import Path

# Define paths to data directories
DATA_DIR = Path("../data/processed")
TRENDS_DIR = DATA_DIR / "google-trends-marketing-v2"

#2 Load the Processed CSV File

In [2]:
# Load datasets
museum = pd.read_csv(DATA_DIR / "museum_monthly.csv")
weather = pd.read_csv(DATA_DIR / "weather_monthly.csv")
calendar_monthly = pd.read_csv(DATA_DIR / "calendar_monthly.csv")
calendar_seasonality = pd.read_csv(DATA_DIR / "calendar_seasonality.csv")
trends = pd.read_csv(TRENDS_DIR / "trends_all_collected_monthly_long.csv")

#3 Check the Shape and Columns of Each Dataset

In [3]:
# Print shapes of the datasets
print("Museum shape:", museum.shape)
print("Weather shape:", weather.shape)
print("Calendar Monthly shape:", calendar_monthly.shape)
print("Calendar Seasonality shape:", calendar_seasonality.shape)
print("Trends shape:", trends.shape)

Museum shape: (90, 2)
Weather shape: (90, 6)
Calendar Monthly shape: (90, 13)
Calendar Seasonality shape: (90, 11)
Trends shape: (8598, 8)


In [4]:
# Print column names of the datasets
print("Museum columns:", museum.columns.tolist())
print("Weather columns:", weather.columns.tolist())
print("Calendar Monthly columns:", calendar_monthly.columns.tolist())
print("Calendar Seasonality columns:", calendar_seasonality.columns.tolist())
print("Trends columns:", trends.columns.tolist())

Museum columns: ['month', 'visitors']
Weather columns: ['month', 'avg_temp_F', 'min_temp_F', 'max_temp_F', 'total_precip_in', 'avg_wind_mph']
Calendar Monthly columns: ['month', 'year', 'month_num', 'quarter', 'is_winter', 'is_spring', 'is_summer', 'is_fall', 'holiday_season', 'summer_tourism_season', 'school_year_month', 'month_sin', 'month_cos']
Calendar Seasonality columns: ['month', 'year', 'month_num', 'has_federal_holiday', 'mn_only_holiday', 'holiday_season', 'summer_season', 'spring_break', 'fall_break', 'winter_break', 'la_event_season']
Trends columns: ['month', 'value', 'term', 'lane', 'theme', 'geo', 'source', 'notes']


#4 Check if Trends File Is In Long Format

In [5]:
# Display first few rows of the trends dataset
trends.head()

,month,value,term,lane,theme,geo,source,notes
0,2014-01,0,Avila Adobe,museum_specific,museum_brand,US-CA,Google Trends via pytrends,Dataset venue; brand search
1,2014-02,0,Avila Adobe,museum_specific,museum_brand,US-CA,Google Trends via pytrends,Dataset venue; brand search
2,2014-03,0,Avila Adobe,museum_specific,museum_brand,US-CA,Google Trends via pytrends,Dataset venue; brand search
3,2014-04,0,Avila Adobe,museum_specific,museum_brand,US-CA,Google Trends via pytrends,Dataset venue; brand search
4,2014-05,0,Avila Adobe,museum_specific,museum_brand,US-CA,Google Trends via pytrends,Dataset venue; brand search


#5 Convert Trends File to Wide Format

In [6]:
# Pivot the trends dataset to have one column per term
trends_wide = trends.pivot_table(
    index="month",
    columns="term",
    values="value"
).reset_index()

# Remove the column grouping name
trends_wide.columns.name = None

trends_wide.head()

,month,Avila Adobe,El Pueblo Los Angeles,Getty Museum,Griffith Observatory,La Brea Tar Pits,Los Angeles attractions,Los Angeles history,Los Angeles museums,Mexican heritage Los Angeles,...,things to do in LA for 2 hours,things to do in LA for a couple hours,things to do in LA in December,things to do in LA in February,things to do in LA in January,things to do in LA when it rains,things to do in Los Angeles,things to do in Los Angeles in winter,things to do in downtown LA,things to do near Union Station Los Angeles
0,2014-01,0.0,48.0,78.0,55.0,58.0,70.0,91.0,81.0,0.0,...,0.0,0.0,0.0,37.0,69.0,0.0,52.0,0.0,38.0,0.0
1,2014-02,0.0,44.0,64.0,47.0,56.0,61.0,82.0,63.0,0.0,...,0.0,0.0,0.0,59.0,0.0,0.0,54.0,0.0,34.0,0.0
2,2014-03,0.0,56.0,74.0,52.0,65.0,72.0,94.0,66.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,58.0,0.0,31.0,0.0
3,2014-04,0.0,61.0,76.0,78.0,60.0,73.0,97.0,61.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,50.0,0.0,32.0,0.0
4,2014-05,0.0,71.0,69.0,53.0,59.0,64.0,89.0,50.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,55.0,0.0,44.0,0.0


In [7]:
# Print shape and columns of the pivoted trends dataset
print(trends_wide.shape)
print(trends_wide.columns.tolist())

(144, 44)
['month', 'Avila Adobe', 'El Pueblo Los Angeles', 'Getty Museum', 'Griffith Observatory', 'La Brea Tar Pits', 'Los Angeles attractions', 'Los Angeles history', 'Los Angeles museums', 'Mexican heritage Los Angeles', 'Olvera Street', 'cultural experiences Los Angeles', 'cultural heritage Los Angeles', 'family activities Los Angeles', 'family things to do in downtown LA', 'free museums Los Angeles', 'heritage tourism Los Angeles', 'historic attractions Los Angeles', 'historic district Los Angeles', 'historic house Los Angeles', 'historic house museum', 'historic sites California', 'historical sites Los Angeles', 'indoor things to do in Los Angeles', 'interactive museums Los Angeles', 'kid friendly museums Los Angeles', 'life in the 1840s', 'living history museum', 'oldest building in Los Angeles', 'oldest home in Los Angeles', 'oldest house Los Angeles', 'oldest house in Los Angeles', 'rainy day activities Los Angeles', 'things to do in LA', 'things to do in LA for 2 hours', 'th

In [8]:
# Clean column names: replace spaces with underscores and convert to lowercase
trends_wide.columns = [
    col.replace(" ", "_").lower() if col != "month" else col
    for col in trends_wide.columns
]

#6 Standardize the Month Column in All Datasets

In [9]:
# Ensure the 'month' column is in datetime format and then convert to 'YYYY-MM' string format
for df in [museum, weather, calendar_monthly,calendar_seasonality, trends_wide]:
    df["month"] = pd.to_datetime(df["month"]).dt.strftime("%Y-%m")

#7 Check for Duplicate Months

In [10]:
# Check for duplicates in the 'month' column of each dataset
print("Museum duplicate months:", museum["month"].duplicated().sum())
print("Weather duplicate months:", weather["month"].duplicated().sum())
print("Calendar Monthly duplicate months:", calendar_monthly["month"].duplicated().sum())
print("Calendar Seasonality duplicate months:", calendar_seasonality["month"].duplicated().sum())
print("Trends duplicate months:", trends_wide["month"].duplicated().sum())

Museum duplicate months: 0
Weather duplicate months: 0
Calendar Monthly duplicate months: 0
Calendar Seasonality duplicate months: 0
Trends duplicate months: 0


#8 Merge All Datasets Into One Final Dataset

In [11]:
# Merge datasets on 'month' column
final_df = museum.merge(weather, on="month", how="left")
final_df = final_df.merge(trends_wide, on="month", how="left")
final_df = final_df.merge(calendar_monthly, on="month", how="left")
final_df = final_df.merge(calendar_seasonality, on="month", how="left")

#9 Inspect the Merged Dataset and Check for Missing Values

In [12]:
# Print shape and columns of the final merged dataset
print("Final dataset shape:", final_df.shape)
print("Final dataset columns:", final_df.columns.tolist())
final_df.head()

# Check for missing values in the final dataset
final_df.isna().sum()

Final dataset shape: (90, 72)
Final dataset columns: ['month', 'visitors', 'avg_temp_F', 'min_temp_F', 'max_temp_F', 'total_precip_in', 'avg_wind_mph', 'avila_adobe', 'el_pueblo_los_angeles', 'getty_museum', 'griffith_observatory', 'la_brea_tar_pits', 'los_angeles_attractions', 'los_angeles_history', 'los_angeles_museums', 'mexican_heritage_los_angeles', 'olvera_street', 'cultural_experiences_los_angeles', 'cultural_heritage_los_angeles', 'family_activities_los_angeles', 'family_things_to_do_in_downtown_la', 'free_museums_los_angeles', 'heritage_tourism_los_angeles', 'historic_attractions_los_angeles', 'historic_district_los_angeles', 'historic_house_los_angeles', 'historic_house_museum', 'historic_sites_california', 'historical_sites_los_angeles', 'indoor_things_to_do_in_los_angeles', 'interactive_museums_los_angeles', 'kid_friendly_museums_los_angeles', 'life_in_the_1840s', 'living_history_museum', 'oldest_building_in_los_angeles', 'oldest_home_in_los_angeles', 'oldest_house_los_ange

month              0
visitors           0
avg_temp_F         0
min_temp_F         0
max_temp_F         0
                  ..
summer_season      0
spring_break       0
fall_break         0
winter_break       0
la_event_season    0
Length: 72, dtype: int64

Save the Merged Dataset

In [13]:
# Save the final dataset to a CSV file
final_df.to_csv(DATA_DIR / "final_dataset.csv", index=False)
print("Saved final_dataset.csv")

Saved final_dataset.csv
